# Fig S6B — GTEx EUR simulation (data processing)

GTEx EUR eQTL simulation (n = 800) with in-sample, ADSP, UKB and 1000 Genomes LD. SuSiE-rRSS is built per reference from the EB-mix fit, falling back to SER when the reliability flag is set. Bootstrap 95% CIs over replicates.

**Output:** `figure_data/FigS6B.GTEx_EUR_Benchmark.rds` — SuSiE-RSS vs. SuSiE-rRSS by reference × NCP × number of causal variants

In [1]:
source("../../R/utils.R")

n_GWAS <- 800
refs   <- c("insample", "ADSP", "UKB", "kg")

In [2]:
## ---- build SuSiE-rRSS hybrid per reference ---------------------------------
res <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_EUR_fine/summary/result_susie_bias_summary_summary.rds") %>%
  mutate(
    reference = str_extract(method, paste0("^(", paste(refs, collapse = "|"), ")")),
    variant   = str_remove(method, paste0("^(", paste(refs, collapse = "|"), ")")),
    variant   = ifelse(variant == "", "baseline", str_remove(variant, "^_"))   # baseline / SER / mix / mix_SER
  )

## decision from the _mix row: FALSE flag -> keep mix; TRUE -> fall back to SER
dec <- res %>% filter(variant == "mix") %>%
  transmute(input_file, ncausal, reference, use_mix = R_reliability_flag %in% FALSE)

ser_variant <- "SER"      # <-- plain {ref}_SER. Change to "mix_SER" for the EB-mix SER fallback.

hybrid <- bind_rows(
  res %>% filter(variant == "mix")         %>% inner_join(filter(dec,  use_mix), by = c("input_file","ncausal","reference")),
  res %>% filter(variant == ser_variant)   %>% inner_join(filter(dec, !use_mix), by = c("input_file","ncausal","reference"))
) %>% mutate(method = paste0(reference, "_2.0"), variant = "2.0") %>% select(-use_mix)

res2 <- bind_rows(res, hybrid) %>%
  mutate(h2g = as.numeric(str_extract(input_file, "(?<=h2g_)[0-9.]+")), causal = ncausal)

In [3]:
## ---- bootstrap over replicate rows within each cell (boot_cell_mix() in R/utils.R)
set.seed(1)
summ <- res2 %>%
  group_by(method, reference, variant, h2g, causal) %>%
  group_modify(~ boot_cell_mix(.x, 1000)) %>% ungroup()

## SuSiE-RSS vs. SuSiE-rRSS — rows = reference, cols = NCP, x = # causal variants

In [5]:
ref_lab <- c(insample = "In-sample", ADSP = "ADSP", UKB = "UKB", kg = "1000 Genomes")

pl_bench <- summ %>%
  filter(variant %in% c("baseline", "2.0")) %>%
  mutate(
    ncp_num   = n_GWAS * as.numeric(h2g),
    ncp       = fct_reorder(
      paste0("NCP = ", formatC(ncp_num, format = "f", digits = 0)),
      ncp_num
    ),
    reference = factor(ref_lab[reference], levels = unname(ref_lab)),
    variant   = factor(
      variant,
      levels = c("2.0", "baseline"),
      labels = c("SuSiE-rRSS", "SuSiE-RSS")
    ),
    causal = factor(causal)
  ) %>%
  transmute(
    reference, variant, ncp, causal,
    Power_est = power, Power_lo = power_lo, Power_hi = power_hi,
    FDR_est = FDR, FDR_lo = FDR_lo, FDR_hi = FDR_hi
  ) %>%
  pivot_longer(
    -c(reference, variant, ncp, causal),
    names_to = c("metric", ".value"),
    names_sep = "_"
  ) %>%
  mutate(metric = factor(metric, levels = c("Power", "FDR")))

save_data(pl_bench, "FigS6B.GTEx_EUR_Benchmark")